# Notebook 1: Data Understanding, Parsing & Structural Validation

**BT4222 Project - JDSearch (e-commerce personalised product ranking)**

This notebook executes the first pipeline stage as defined in the project guidelines and the interim report. 

**Objectives:**
1. Load the raw, tab-separated `user_behavior_data.txt` file.
2. Parse the complex `_`-delimited list columns into usable Python lists.
3. Validate the structural integrity of the parallel history lists.
4. Explode the session records into two interaction-level tables:
   - `candidate_interactions_df`: The unit of prediction (one row per user-candidate pair).
   - `historical_interactions_df`: The chronological history of user actions.
5. Quantify data sparsity, cold-start distributions, and timing gaps.
6. Save these foundational artifacts to disk for downstream modeling.


## 1. Environment Setup & Data Loading

We load the raw data using `pandas`, ensuring `dtype=str` to preserve the integrity of the anonymized IDs and `low_memory=False` to prevent mixed-type warnings during parsing.


In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import chain
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

# Define Paths
BASE = Path(r"C:\Users\Acer\OneDrive - National University of Singapore\BT4222\JDsearch")
USER_TXT = BASE / "user_behavior_data.txt"
ARTIFACT_DIR = BASE / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading user_behavior_data.txt...")
user_df = pd.read_csv(USER_TXT, sep="\t", dtype=str, low_memory=False)

print(f"Raw Dataset Shape: {user_df.shape}")
display(user_df.head(3))

## 2. List Parsing & Feature Extraction

The raw file stores variable-length lists as `_`-delimited strings. 
A single history row has three parallel lists of equal length (`qry`, `wid`, `type`), plus one time list that is exactly **one element longer** (a leading `0` and a trailing gap to the test query). We parse these into Python lists.


In [ ]:
def safe_split(val, sep="_"):
    if pd.isna(val) or str(val).strip() == "": return []
    return str(val).split(sep)

def safe_split_numeric(val, sep="_"):
    return [float(x) for x in safe_split(val, sep)]

# Parse Candidate Lists
user_df["candidate_wid_list_parsed"]   = user_df["candidate_wid_list"].apply(safe_split_numeric)
user_df["candidate_label_list_parsed"] = user_df["candidate_label_list"].apply(safe_split_numeric)

# Parse History Lists
user_df["history_qry_list_parsed"]  = user_df["history_qry_list"].apply(safe_split)
user_df["history_wid_list_parsed"]  = user_df["history_wid_list"].apply(safe_split)
user_df["history_type_list_parsed"] = user_df["history_type_list"].apply(safe_split)
user_df["history_time_list_parsed"] = user_df["history_time_list"].apply(safe_split_numeric)

print("List parsing complete.")
display(user_df[["candidate_wid_list_parsed", "history_wid_list_parsed"]].head(3))

## 3. Structural Integrity Checks

Before exploding the data, we must verify the parallel-list constraints. We check that `len(qry) == len(wid) == len(type) == len(time) - 1` for every row. We also verify that no candidate appears twice in a single slate, and that the test query rarely appears in the user's own history.


In [ ]:
def check_lengths(row):
    lq  = len(row["history_qry_list_parsed"])
    lw  = len(row["history_wid_list_parsed"])
    lt  = len(row["history_type_list_parsed"])
    ltm = len(row["history_time_list_parsed"])
    return (lq == lw == lt) and (lw == ltm - 1)

user_df["length_ok"] = user_df.apply(check_lengths, axis=1)
n_ok    = int(user_df["length_ok"].sum())
n_total = len(user_df)

def count_repeats(lst):
    return 0 if not lst else len(lst) - len(set(lst))

dup_candidates = user_df["candidate_wid_list_parsed"].apply(count_repeats)
n_dups = int((dup_candidates > 0).sum())

def query_in_history(row):
    return row["query"] in row["history_qry_list_parsed"]

n_in_hist = int(user_df.apply(query_in_history, axis=1).sum())

print("=== Data Integrity Report ===")
print(f"Total Rows Analyzed: {n_total:,}")
print(f"Rows passing length integrity check: {n_ok:,} ({n_ok/n_total:.2%})")
print(f"Rows with duplicate candidate IDs in one slate: {n_dups:,}")
print(f"Rows where test query appears in own history: {n_in_hist:,}")

## 4. Explode to Candidate-Level Interactions

The unit of prediction in the project is a **(session, candidate)** pair. Every candidate was by definition displayed, so `is_recommended = 1`. We map the integer labels to descriptive action types:
- `0.0` -> No Interaction
- `1.0` -> Click
- `2.0` -> Add to Cart
- `3.0` -> Purchase


In [ ]:
SESSION_COL = "session_id"
user_df_r = user_df.reset_index().rename(columns={"index": SESSION_COL})

cand_df = user_df_r[[SESSION_COL, "query", "candidate_wid_list_parsed", "candidate_label_list_parsed"]].copy()
cand_df = cand_df.explode(["candidate_wid_list_parsed", "candidate_label_list_parsed"])
cand_df = cand_df.rename(columns={"query": "query_id", "candidate_wid_list_parsed": "item_id", "candidate_label_list_parsed": "raw_action_label"})
cand_df["item_id"] = cand_df["item_id"].astype("Int64")
cand_df["is_recommended"] = 1

label_map = {0.0: "No Interaction", 1.0: "Click", 2.0: "Cart", 3.0: "Purchase"}
cand_df["action_type"] = cand_df["raw_action_label"].map(label_map)

cand_df = cand_df[[SESSION_COL, "query_id", "item_id", "is_recommended", "action_type", "raw_action_label"]]

print(f"Candidate Interactions Shape: {cand_df.shape}")
display(cand_df.head(3))

## 5. Explode to Historical Interactions

We zip the three equal-length history lists with `history_time_list[1:-1]` to keep the times aligned with the events they describe. The interim report noted the sample notebook's `is_recommended` flag was ambiguous; we use the unambiguous `is_queryless` flag instead (where `query_id == "-1"`).


In [ ]:
hist_df = user_df_r[[SESSION_COL, "history_qry_list_parsed", "history_wid_list_parsed", "history_type_list_parsed", "history_time_list_parsed"]].copy()

def zip_history(row):
    qs, ws, ts, tms = row["history_qry_list_parsed"], row["history_wid_list_parsed"], row["history_type_list_parsed"], row["history_time_list_parsed"]
    tms_aligned = tms[1:-1] if len(tms) == len(qs) + 1 else tms
    return list(zip(qs, ws, ts, tms_aligned))

hist_df["zipped_history"] = hist_df.apply(zip_history, axis=1)
hist_df = hist_df.explode("zipped_history")

hist_df[["query_id", "item_id", "action_type", "time_to_next"]] = pd.DataFrame(hist_df["zipped_history"].tolist(), index=hist_df.index)
hist_df["item_id"] = pd.to_numeric(hist_df["item_id"], errors="coerce").astype("Int64")
hist_df["query_id"] = hist_df["query_id"].astype(str)
hist_df["is_queryless"] = (hist_df["query_id"] == "-1").astype(int)

hist_df = hist_df.drop(columns=["history_qry_list_parsed", "history_wid_list_parsed", "history_type_list_parsed", "history_time_list_parsed", "zipped_history"])
hist_df = hist_df[[SESSION_COL, "query_id", "item_id", "is_queryless", "action_type", "time_to_next"]]

print(f"Historical Interactions Shape: {hist_df.shape}")
display(hist_df.head(3))

## 6. Descriptive Statistics: Sparsity, Cold-Start & Timing

Here we reproduce the exact distributions detailed in the interim report to validate our parsing.


In [ ]:
print("=== Candidate Label Distribution ===")
lab_counts = cand_df["raw_action_label"].value_counts(dropna=False).sort_index()
lab_pct    = (lab_counts / lab_counts.sum() * 100).round(2)
for k, v in lab_counts.items():
    print(f"  Label {k}: {v:,} ({lab_pct.loc[k]}%)")

print("\n=== Slate Size (Candidates per Session) ===")
slate_sizes = user_df["candidate_label_list_parsed"].apply(len)
display(slate_sizes.describe(percentiles=[.25, .5, .75, .9, .95]))

print("\n=== History Length (Events per Session) ===")
hist_lengths = user_df["history_wid_list_parsed"].apply(len)
display(hist_lengths.describe(percentiles=[.25, .5, .75, .9, .95]))

In [ ]:
print("=== Cold Start Analysis ===")
COLD_USER_THRESHOLD = 3
n_cold_users = int((hist_lengths < COLD_USER_THRESHOLD).sum())
print(f"Cold Users (<{COLD_USER_THRESHOLD} history events): {n_cold_users:,} ({n_cold_users/len(user_df):.2%})")
print(f"Active Users: {len(user_df) - n_cold_users:,} ({1 - n_cold_users/len(user_df):.2%})")

global_history_items = set(chain.from_iterable(user_df["history_wid_list_parsed"]))
global_history_int   = {int(x) for x in global_history_items if str(x).isdigit()}
print(f"Distinct items in all supplied histories: {len(global_history_int):,}")

all_candidates      = user_df["candidate_wid_list_parsed"].explode().dropna().astype(int)
unique_candidates   = all_candidates.unique()
print(f"Distinct candidate products: {len(unique_candidates):,}")

cold_mask = ~np.isin(unique_candidates, np.fromiter(global_history_int, dtype=np.int64))
n_cold    = int(cold_mask.sum())
n_warm    = len(unique_candidates) - n_cold
print(f"Cold Candidate Products (Never in History): {n_cold:,} ({n_cold/len(unique_candidates):.2%})")
print(f"Warm Candidate Products: {n_warm:,} ({n_warm/len(unique_candidates):.2%})")

## 7. Save Artifacts for Downstream Modeling

We save the interaction-level tables to Parquet. This step is critical: if this cell does not execute successfully, Notebook 2 will not have the data it needs to run.


In [ ]:
cand_path = ARTIFACT_DIR / "candidate_interactions.parquet"
hist_path = ARTIFACT_DIR / "historical_interactions.parquet"

cand_df.to_parquet(cand_path, index=False)
hist_df.to_parquet(hist_path, index=False)

user_summary = pd.DataFrame({
    "session_id": user_df_r[SESSION_COL].values,
    "history_length": hist_lengths.values,
    "slate_size": slate_sizes.values,
    "gap_to_test_query": user_df["history_time_list_parsed"].apply(lambda t: t[-1] if len(t) > 0 else np.nan).values,
})
user_summary.to_parquet(ARTIFACT_DIR / "user_session_summary.parquet", index=False)

print("=== Artifacts Saved Successfully ===")
for p in ARTIFACT_DIR.glob("*.parquet"):
    print(f"{p.name:35s} {p.stat().st_size/1e6:8.2f} MB")